### The DeepRitz-Method

Consider again the Poisson equation, but with a singularity at (0, 0):
\begin{align*}
    -\Delta u &= 1 \text{ in } \Omega, \\
    u &= 0 \text{ on } \partial \Omega,
\end{align*} 

with $\Omega=([-1, 1] \times [-1, 1]) \setminus ([0, 1] \times \{0\})$.

Then the energy functional for this problem is given by:
\begin{equation}
    E(v) = \int_\Omega \frac{1}{2}\|\nabla v\|^2 \text{d}x.
\end{equation}
The solution $u$ of the strong formulation minimizes the energy functional, namely $E(u) \leq E(v)$ for all $v$ in the corresponding function space.

In practice, we need to include the boundary condition and minimize the functional:
\begin{equation}
    \tilde{E}(v) = w_1\int_\Omega \frac{1}{2}\|\nabla v\|^2 \text{d}x + w_2\int_{\partial\Omega} \frac{1}{2}\|v\|^2 \text{d}x.
\end{equation}

In [ ]:
import torchphysics as tp 
import torch
import pytorch_lightning as pl

# This time, we create spaces for first and second spatial coordinates separately. This is helpful later for an easy creation of the line [0,1]x{0}
X = tp.spaces.R1('x') 
Y = tp.spaces.R1('y')
U = tp.spaces.R1('u')

In [2]:
# TODO: Create square [-1, 1] x [-1, 1] and the line [0,1]x{0} as two domains.
square = tp.domains.Parallelogram(X*Y, ...)
# The line is a cartesian product of an interval and a Point, realized in TorchPhysics by domain1 * domain2
line = tp.domains.Interval(...) * tp.domains.Point(...)

In [10]:
# Here we create the sampler.
inner_sampler = tp.samplers.RandomUniformSampler(square, n_points=100000) 

# For the boundary, we create two samplers. One for the boundary of the square. And one for the line.
bound_sampler_square = tp.samplers.RandomUniformSampler(square.boundary, n_points=40000)
# TODO: Create a sampler, which samples 10000 points on the line [0,1]x{0}
bound_sampler_line = ...
# Samplers can be "added": The sampler below uses both samplers above and returns the union of all sampled points.
bound_sampler = bound_sampler_square + bound_sampler_line

The classic DeepRitz method uses a ResNet-architecture instead of a FCN. This is also implemented and used here: 

In [11]:
# Add the input and output space
model = tp.models.DeepRitzNet(input_space=X*Y, output_space=U, width=20, depth=4)

The transformation of the mathematical equations is handled similar to the PINN approach. For each integral in the energy functional, we define a condition.

Instead of a *PINNCondition*, we now use a *DeepRitzCondition*. While the *PINNCondition* computes the mean of the squared outputs of the residual function, the *DeepRitzCondition* only averages the output of the residual function (without computing squares, which allows negative losses).

In [16]:
# TODO: Define the integrand function for the boundary integral
def bound_residual(...):
    return ...
# TODO: Create a corresponding DeepRitzCondition with weight=100
bound_cond = tp.conditions.DeepRitzCondition(...)

In [17]:
# The integrand function for the integral over Omega.
def energy_residual(u, x, y):
    grad_term = torch.sum(tp.utils.grad(u, x, y)**2, dim=1, keepdim=True)
    return 0.5*grad_term - u

pde_cond = tp.conditions.DeepRitzCondition(module=model, sampler=inner_sampler, 
                                           integrand_fn=energy_residual, weight=1.0)

In [ ]:
learning_rate = 1e-3
training_iterations = 2000

optim = tp.OptimizerSetting(optimizer_class=torch.optim.Adam, lr=learning_rate)
solver = tp.solver.Solver(train_conditions=[bound_cond, pde_cond], optimizer_setting=optim)

trainer = pl.Trainer(devices=1, accelerator="gpu",
                     max_steps=training_iterations,
                     logger=False,
                     benchmark=True,
                     enable_checkpointing=False)

trainer.fit(solver)

In [ ]:
plot_sampler = tp.samplers.PlotSampler(plot_domain=square, n_points=640, device='cuda')
fig = tp.utils.plot(model, lambda u : u, plot_sampler, plot_type='contour_surface')

We can also use different optimizer algorithms, like LBFGS to may get better convergence.

One point we have to keep in mind, is to fix the inputs of the neural network for LBFGS to work.

In [ ]:
optim = tp.OptimizerSetting(optimizer_class=torch.optim.LBFGS, lr=0.05, 
                            optimizer_args={'max_iter': 2, 'history_size': 100})

pde_cond.sampler = pde_cond.sampler.make_static() 
bound_cond.sampler = bound_cond.sampler.make_static() 
solver = tp.solver.Solver(train_conditions=[bound_cond, pde_cond], optimizer_setting=optim)

trainer = pl.Trainer(devices=1, accelerator="gpu",
                     max_steps=training_iterations,
                     logger=False,
                     benchmark=True,
                     enable_checkpointing=False)

trainer.fit(solver)

In [ ]:
plot_sampler = tp.samplers.PlotSampler(plot_domain=square, n_points=640, device='cuda')
fig = tp.utils.plot(model, lambda u : u, plot_sampler, plot_type='contour_surface')